# Notebook Analyse Textsemantic

Ce Notebook guide l'analyse des potentiels évoqués (ERP) pour le paradigme de lecture de phrases (texte congruent vs incongruent).

### Version 2

Il s'agit de la deuxième version de ce notebook. Elle inclut quelques modifications et correctifs pour résoudre des erreurs mineures. Le flux de travail général reste identique à la version précédente.

### Remarque importante

La démarche de ce notebook consiste d'abord à explorer et tester le code **individuellement sur un seul sujet**.

Une fois les étapes validées, une **fonction finale** regroupe tous les blocs de code. Cette fonction permet d'appliquer l'ensemble du pipeline de traitement à **tous les sujets** de manière automatisée.

> **Attention :** Si vous avez expérimenté avec différents paramètres (par exemple, autre ERP ou differente façon de définir les piques) et que vous souhaitez utiliser des valeurs autres que celles définies par défaut, **ou si vous avez ajouté d'autres blocs de code ou de nouvelles méthodes**, n'oubliez pas de **mettre à jour la fonction finale** en conséquence avant de l'exécuter sur l'ensemble des données.

Le notebook reprend la structure "Type 1 / Type 2 / Type 3" introduite dans `01_preprocessing_notebook.ipynb` :
- **Type 1 — prêt à exécuter** : cellules complètes, sans modification nécessaire.
- **Type 2 — à personnaliser** : blocs contenant des paramètres à ajuster (balises `A_COMPLETER`).
- **Type 3 — exploration libre** : propositions d'analyses supplémentaires, prêtes à modifier.

## Pour bien démarrer
- Assurez-vous d'avoir exécuté le pipeline de prétraitement pour générer les fichiers `*_clean_(eeg ou epo).fif`.
- Activez l'environnement Python du cours et installez les dépendances (`pip install -r requirements.txt`).

## Objectifs pédagogiques
1. Charger un enregistrement textsemantic déjà prétraité et vérifier ses métadonnées.
2. Extraire les événements congruent / incongruent, créer des epochs équilibrés et visualiser les ERP.
3. Mesurer la N400 et P600 (amplitude/latence) au niveau sujet et au niveau groupe, en sauvegardant les résultats pour réutilisation ultérieure.


## 0. Préparation et configuration

Nous configurons l’environnement d’analyse : imports, chemins BIDS, sélection du sujet. Chaque bloc détaille ce qu’il fait et comment vous pouvez l’adapter pour vos explorations (ex. utiliser un autre chemin, ajouter un sujet, jouer avec les paramètres de rejet).


### Bloc Type 1 — Imports et options globales

Charge les bibliothèques principales (MNE, NumPy, Matplotlib, outils BIDS) et fixe quelques réglages d’affichage. Si vous expérimentez d’autres packages (par ex. `seaborn` ou `scipy`), importez-les ici pour garder le carnet cohérent.


In [ ]:
# -----------------------------------------------------------------------------
# Imports principaux et configuration globale
# -----------------------------------------------------------------------------
import warnings  # contrôle des avertissements Python
from pathlib import Path  # manipulation de chemins indépendante de l'OS
import csv  # lecture / écriture de fichiers tabulés (participants.tsv)
import json  # sauvegarde des métriques intermédiaires
from collections import Counter  # comptage rapide des annotations

import matplotlib.pyplot as plt  # visualisation des ERP
import mne  # bibliothèque principale pour l'analyse EEG/MEG
import numpy as np  # calcul scientifique vectorisé
import mne_bids  # accès à la version et aux outils BIDS
from mne_bids import BIDSPath  # gestion de chemins compatibles BIDS

warnings.filterwarnings('ignore', category=RuntimeWarning)  # masque certains warnings MNE
mne.set_log_level('INFO')  # verbosité modérée pour suivre les étapes clés
plt.rcParams['figure.figsize'] = (10, 5)  # taille par défaut des figures

print('Versions utilisées:')
print(' - mne      ', mne.__version__)
print(' - numpy    ', np.__version__)
print(' - mne_bids ', mne_bids.__version__)


### Bloc Type 1 — Définir le dossier BIDS et les dérivés

Localise les données brutes et configure les dossiers de sortie :
- `derivatives/preproc` contient les fichiers nettoyés produits par le notebook 01 (AutoReject + ICA).
- `derivatives/textsemantic-erp` regroupe les ERP de groupe fournis avec le dataset (optionnel).
- `derivatives/textsemantic-analysis` accueille les artefacts générés ici (métriques JSON, difference waves, exports CSV). Vous pouvez créer d’autres sous-dossiers si vous développez de nouvelles analyses.


In [ ]:
# -----------------------------------------------------------------------------
# Localisation des données BIDS et des dérivés nécessaires
# -----------------------------------------------------------------------------
root_bids = Path('tasks/text_reading/bids')

print('Chemins vérifiés:')  # confirmation dans la console
print(' - BIDS root            :', root_bids.resolve())  # chemin absolu utilisé

# Dérivés harmonisés avec le notebook de prétraitement
deriv_preproc = root_bids / 'derivatives' / 'preproc'
deriv_preproc.mkdir(parents=True, exist_ok=True)

# Dérivés spécifiques à cette analyse
deriv_erp = root_bids / 'derivatives' / 'textsemantic-erp'
deriv_analysis = root_bids / 'derivatives' / 'textsemantic-analysis'
deriv_analysis.mkdir(parents=True, exist_ok=True)

TASK_LABEL = 'textsemantic'
FOCUS_CONDITIONS = ['cw_cong', 'cw_incong']

print('Chemins vérifiés:')
print(' - BIDS root            :', root_bids.resolve())
print(' - dérivés préproc      :', deriv_preproc.resolve())
print(' - dérivés analyse      :', deriv_analysis.resolve())
print(' - dérivés ERP (option) :', deriv_erp.resolve())


### Bloc Type 1 — Lister les participants disponibles

Lit `participants.tsv` (convention BIDS) pour récupérer les identifiants `sub-XX` disponibles. 
En Type 2 ci-dessous, nous choisirons l'un de ces sujets.

In [ ]:
# -----------------------------------------------------------------------------
# Lecture de participants.tsv afin d'obtenir la liste des sujets présents
# -----------------------------------------------------------------------------
participants_tsv = root_bids / 'participants.tsv'  # chemin vers le fichier BIDS
subjects = []  # contiendra les identifiants sans le préfixe 'sub-'

with participants_tsv.open('r', encoding='utf-8') as f:  # ouverture du fichier en lecture
    reader = csv.reader(f, delimiter='	')  # lecture tabulée
    header = next(reader, None)  # saute l'entête (participant_id, ...)
    for row in reader:  # boucle sur chaque ligne restante
        if not row:  # ignore les lignes vides
            continue
        participant_id = row[0]  # première colonne = identifiant sujet
        if participant_id.startswith('sub-'):  # vérifie le format BIDS
            subjects.append(participant_id.replace('sub-', ''))  # stocke l'identifiant sans préfixe

print(f"Participants détectés ({len(subjects)}): {subjects}")  # affiche la liste obtenue

### Bloc Type 2 — Sélectionner un participant et une session

Modifiez les identifiants ci-dessous pour analyser un autre sujet. Chaque sujet dispose d'un unique run (`run = '01'`).

In [ ]:
# -----------------------------------------------------------------------------
# Choix du sujet / session / run à analyser
# -----------------------------------------------------------------------------
subject = '10'  # <-- remplacez par un identifiant présent dans `subjects`
session = '001'  # la plupart des sujets possèdent cette session unique
run = '01'  # un seul run textsemantic est disponible

print(f'Sujet en cours: sub-{subject}, session {session}, run {run}')


## 1. Charger un enregistrement prétraité

Nous chargeons le fichier `*_clean.fif` issu du pipeline précédent, appliquons un montage standard et vérifions les métadonnées clés.

### Bloc Type 1 — Fonction utilitaire de chargement

`load_processed_raw` centralise la construction du chemin de fichier, le chargement `Raw` MNE et l'application d'un montage 10-20 international.

In [ ]:
# -----------------------------------------------------------------------------
# Fonction utilitaire : chargement d'un fichier prétraité pour un sujet donné
# -----------------------------------------------------------------------------
def load_processed_raw(subject: str, session: str = '001', run: str = '01') -> mne.io.BaseRaw:
    # Construit le chemin BIDS du fichier *_processed.fif dans derivatives/preproc
    processed_bids = BIDSPath(
        root=deriv_preproc,
        subject=subject,
        session=session,
        task='textsemantic',
        run=run,
        datatype='eeg',
        suffix='eeg',
        processing='clean',
        extension='.fif'
    )
    fname = processed_bids.fpath
    if not fname.exists():  # garde-fou si le fichier manque
        raise FileNotFoundError(f'Fichier introuvable: {fname}')  # message explicite
    raw_obj = mne.io.read_raw_fif(fname, preload=True)  # charge en mémoire pour un accès rapide
    raw_obj.set_montage('standard_1020', match_case=False, on_missing='warn')  # assure la co-registration EEG
    return raw_obj  # renvoie l'objet Raw prêt à l'emploi

raw = load_processed_raw(subject, session=session, run=run)  # chargement effectif
print(raw)  # résumé de l'objet Raw


### Bloc Type 1 — Vérifier les annotations et métadonnées

Affiche la fréquence d'échantillonnage, la liste des canaux EEG, ainsi que les annotations importées (événements détectés et autres marquages).


In [ ]:
raw.plot(start=165, duration=5)  # décommentez pour inspecter visuellement

In [ ]:
# -----------------------------------------------------------------------------
# Inspection rapide des métadonnées pour valider le chargement
# -----------------------------------------------------------------------------
print('Fréquence échantillonnage :', raw.info['sfreq'], 'Hz')  # vérifie la fréquence
print('Annotations disponibles   :', sorted(set(raw.annotations.description)))  # types d'événements
print('Nombre total annotations  :', len(raw.annotations))  # quantité d'annotations

### Bloc Type 2 — Visualisation rapide du signal brut (optionnel)

Décommentez la ligne suivante pour afficher quelques secondes de signal. 
Ajustez `n_channels`, `scalings` ou la fenêtre temporelle selon vos besoins.

In [ ]:
# raw.copy().crop(tmax=5).plot(n_channels=12, scalings='auto', title='Brut prétraité (5 s)')


## 2. Préparer les événements et créer les epochs

Nous transformons les annotations textuelles en codes numériques, définissons la fenêtre temporelle des epochs et construisons un jeu d’essais équilibré (congruent vs incongruent) prêt pour les analyses N400.


### Bloc Type 1 — Extraire les événements textsemantic

Nous mappons les annotations BIDS (`Stimulus/S 21`, `Stimulus/S 22`) vers les labels `Congruent` / `Incongruent` via `ANNOTATION_MAP`. Ce bloc vérifie la disponibilité de chaque condition cible.


In [ ]:
# -----------------------------------------------------------------------------
# Extraction des événements et renommage en labels lisibles
# -----------------------------------------------------------------------------
annotation_map = {
    'Stimulus/S 21': 'cw_cong',
    'Stimulus/S 22': 'cw_incong',
}

events, event_id = mne.events_from_annotations(raw)
selected_event_id = {}
for original, label in annotation_map.items():
    if np.str_(label) in event_id:
        selected_event_id[label] = event_id[np.str_(label)]

print('Étiquettes retenues:')
for label, code in selected_event_id.items():
    n_trials = int((events[:, 2] == code).sum())
    print(f' - {label:12s} → code {code:3d}, essais = {n_trials}')

focus_event_id = {label: code for label, code in selected_event_id.items() if label in FOCUS_CONDITIONS}

### Bloc Type 1 — Paramètres d'epoching

Définit la fenêtre temporelle autour des événements et calcule les epochs EEG associés.
- changez la baseline (ex. `(-0.1, 0)`),
- modifiez le seuil `reject` pour observer son impact sur le nombre d’essais conservés.


In [ ]:
# -----------------------------------------------------------------------------
# Paramètres d'epoching et construction des epochs MNE
# -----------------------------------------------------------------------------

# Définition de la fenêtre temporelle pour chaque epoch :
tmin, tmax = -0.2, 0.8  # -200 ms avant l'événement (t=0) et +800 ms après.

# Définition de la période de "ligne de base" (baseline) :
baseline = (-0.2, 0.0)  # Utilise l'intervalle [-200 ms, 0 ms] (pré-stimulus)
                        # La moyenne de cette période sera soustraite de toute l'epoch.

# Création de l'objet Epochs
epochs = mne.Epochs(
    raw,  # Le signal continu (filtré) à découper
    events,  # La matrice des événements
    event_id=selected_event_id, # Le dictionnaire propre
    tmin=tmin,  # Début de la fenêtre
    tmax=tmax,  # Fin de la fenêtre
    baseline=baseline,  # Période de correction de la ligne de base
    picks='eeg',  # Sélectionne UNIQUEMENT les canaux de type 'eeg'
    preload=True,  # Charge toutes les données des epochs en mémoire (RAM).
                  # REQUIS pour AutoReject, ICA, etc.
    detrend=None,  # N'applique pas de "detrending"
)

print(epochs)  # Affiche un résumé (nombre d'epochs, canaux, temps)

# Comptage final des essais pour chaque condition
# (en utilisant les étiquettes de 'selected_event_id')
trial_counts = {cond: len(epochs[cond]) for cond in epochs.event_id}

print('Essais conservés par condition :', trial_counts)  # Affichage console

### Bloc Type 2 — Visualiser quelques epochs (optionnel)

Utilisez ces lignes (commentées) pour inspecter des epochs individuelles. Encouragez les étudiants à chercher des artefacts résiduels ou des patterns particuliers.


In [ ]:
# epochs['cw_cong'][:5].plot(n_channels=15, scalings='auto', title='Epochs congruentes (5 premières)')
# epochs['cw_incong'][:5].plot(n_channels=15, scalings='auto', title='Epochs incongruentes (5 premières)')


## 3. Construire et comparer les ERP

Nous calculons les ERP par condition, examinons leurs topographies et générons la “waveform” de différence (Incongruent – Congruent) typique d’une N400.


### Bloc Type 1 — Calculer les ERP par condition

Moyennage simple des epochs `cw_cong` et `cw_incong`. Les graphiques produits (matplotlib interactive) facilitent la discussion en classe : points d’intérêt, latence du pic négatif, etc.


In [ ]:
# -----------------------------------------------------------------------------
# Calcul des ERP pour chaque condition retenue
# -----------------------------------------------------------------------------
evokeds = {}
for label in focus_event_id:
    if len(epochs[label]) == 0:
        print(f'Avertissement: aucune epoch pour {label}.')
        continue
    evk = epochs[label].average()
    evokeds[label] = evk
    evk.plot(spatial_colors=True, time_unit='s', titles=f'ERP — {label}')


### Bloc Type 2 — Topographies temporelles (fenêtre N400)

Affiche les topomaps entre 300 et 500 ms. Vous pouvez modifier `times` pour explorer d’autres latences (P2 vers 200 ms, LPP après 500 ms) ou ajouter un curseur interactif (`mne.viz.plot_evoked_topomap`).


In [ ]:
# -----------------------------------------------------------------------------
# Topographies entre 300 et 500 ms (fenêtre N400)
# -----------------------------------------------------------------------------
if 'cw_incong' in evokeds:
    evokeds['cw_incong'].plot_topomap(
        times=np.linspace(0.30, 0.50, 6),
        ch_type='eeg',
        time_unit='s',
        colorbar=True,
    )
if 'cw_cong' in evokeds:
    evokeds['cw_cong'].plot_topomap(
        times=np.linspace(0.30, 0.50, 6),
        ch_type='eeg',
        time_unit='s',
        colorbar=True,
    )


### Bloc Type 3 — Onde différence (Incongruent - Congruent)

Calcule l’onde de différence et la sauvegarde (`evoked-ave.fif`). Suggestions d’exploration :
- comparer N400 sur d’autres canaux (ex. `Pz`, `CPz`),
- tester une pondération différente (ex. moyennes pondérées si vos conditions n’ont pas le même nombre d’essais).


In [ ]:
# -----------------------------------------------------------------------------
# Calcul et sauvegarde de l'onde différence (Incongruent - Congruent)
# -----------------------------------------------------------------------------
difference = None
if {'cw_cong', 'cw_incong'}.issubset(evokeds):
    difference = mne.combine_evoked(
        [evokeds['cw_incong'], evokeds['cw_cong']],
        weights=[1, -1]
    )
    difference.plot(spatial_colors=True, time_unit='s', titles='Différence Incongruent - Congruent', gfp=True)

    session_label = session if session is not None else 'NA'
    subject_dir = (deriv_analysis / f'sub-{subject}')
    subject_dir.mkdir(parents=True, exist_ok=True)
    diff_path = subject_dir / f'sub-{subject}_ses-{session_label}_task-{TASK_LABEL}_cond-incong-minus-cong_evoked-ave.fif'
    difference.save(diff_path, overwrite=True)
    print('Onde différence sauvegardée :', diff_path)
else:
    print('Impossible de calculer la différence (ERP manquants).')


### Bloc Type 3 — Visualisations Comparatives, Topographie de la DIFFÉRENCE (optionnel)

Utilisez et adaptez le code en bas pour faire des comparaisons entre conditions spécifiques, entre "cw_cong" et "cw_incong".

In [ ]:
# -----------------------------------------------------------------------------
# Visualisation B : Topographie de la DIFFÉRENCE (Incongruent - Congruent)
# -----------------------------------------------------------------------------
#
# La meilleure façon de comparer est de soustraire un ERP de l'autre.
# Cela crée une "onde de différence" (Difference Wave).
# Nous pouvons ensuite tracer la topographie de cette différence.
t_center = 0.325  # Centre de la fenêtre (325 ms)
t_width = 0.25    # Largeur de la fenêtre (250 ms)
if 'cw_cong' in evokeds and 'cw_incong' in evokeds:
    print("Calcul et affichage de la topographie de la DIFFÉRENCE")
    
    # 1. Créer l'Evoked de différence : Incongruent - Congruent
    evk_diff = mne.combine_evoked(
        [evokeds['cw_incong'], evokeds['cw_cong']],
        weights=[1, -1]  # Poids : +1 pour Incongruent, -1 pour Congruent
    )
    
    evk_diff.comment = 'Difference (Incongruent - Congruent)' # Renommer pour la clarté
    
    # 2. Afficher la topographie MOYENNE de cette différence
    #    (C'est souvent le graphique le plus publié)
    evk_diff.plot_topomap(
        times=t_center, # Centre de la fenêtre (0.325 s)
        average=t_width, # Largeur de la fenêtre (0.25 s)
        ch_type='eeg',
        time_unit='s',
        colorbar=True,
        # 'vlim' est important ici pour centrer la couleur sur 0
        vlim=max(np.abs(evk_diff.data.min()), np.abs(evk_diff.data.max())),
    )
    # 
    
    # 3. (Optionnel) Afficher les instantanés de la différence
    evk_diff.plot_topomap(
        times=times_snapshots, # Les 5 instants
        ch_type='eeg',
        time_unit='s',
        colorbar=True,
        vlim=max(np.abs(evk_diff.data.min()), np.abs(evk_diff.data.max())),
    )

## 4. Mesures temporelles (amplitudes / latences)

Calculer les amplitudes et latences des Evoked pour les utiliser comme attributs (features)

Maintenant que nous avons exploré visuellement nos données en moyennant les essais (avec les objets `Evoked`), nous passons à l'étape cruciale pour le **Machine Learning**.

L'objectif ici c'est de calculer les amplitudes et latences des ERPs qui peuvent être utilisé comme attributs.

Un attribut est une mesure numérique qui résume une information clé sur un essai. Notre hypothèse est que des attributs bien choisis (ex: "l'amplitude de la P3") peuvent aider un modèle à distinguer un essai "cong" d'un essai "incong".

Nous allons utiliser les mêmes dictionnaires de configuration (`CHANNEL_CLUSTERS`, `ERP_MEAN_WINDOWS`, `ERP_PEAK_WINDOWS`) et les fonctions d'aides (`mean_amplitude`, `peak_latency`) que nous avons définis.

**La Sortie :** Le résultat final sera un **DataFrame `pandas`**.
    * Chaque **ligne** représentera un **sujet**.
    * Chaque **colonne** représentera un **attribut** (ex: `N400_peak_amplitude_uV` ou `N400_mean_300_500`).

Ce DataFrame sera la matrice `X` (features) et `y` (labels) que nous utiliserons pour entraîner nos modèles de Machine Learning.

### Bloc Type 1 — Fonctions utilitaires

Fonctions réutilisables pour calculer la moyenne en µV et la latence du pic.

In [ ]:
# -----------------------------------------------------------------------------
# Définition des Régions d'Intérêt (ROI) Spatiales
# -----------------------------------------------------------------------------
# Ce dictionnaire regroupe des canaux EEG en "clusters" logiques.
# Cela permet d'analyser une région (ex: "parietal") plutôt qu'un seul
# canal, ce qui augmente le rapport signal/bruit (SNR).
CHANNEL_CLUSTERS = {
    "midline_fc": ["Fz", "Cz"],
    "centro_parietal": ["Cz", "P3", "P4"],
    "parietal": ["P3", "P4"],
    "occipital": ["O1", "O2"],
}

# -----------------------------------------------------------------------------
# Définition des Caractéristiques (Features) basées sur la MOYENNE
# -----------------------------------------------------------------------------
# C'est une "liste de tâches" pour l'extraction d'amplitude MOYENNE.
# Chaque dictionnaire définit une caractéristique à extraire.
ERP_MEAN_WINDOWS = [
    {"name": "N400_mean_300_500_uV", "cluster": "centro_parietal", "tmin": 0.300, "tmax": 0.500},
    {"name": "P600_mean_600_800_uV", "cluster": "parietal", "tmin": 0.600, "tmax": 0.800},
]

# -----------------------------------------------------------------------------
# Définition des Caractéristiques (Features) basées sur le PIC
# -----------------------------------------------------------------------------
# Liste de tâches similaire, mais pour l'extraction de PIC (amplitude ET latence).
ERP_PEAK_WINDOWS = [
    {"name": "N400_peak_300_500", "cluster": "centro_parietal", "tmin": 0.300, "tmax": 0.500, "mode": "min"},
    {"name": "P600_peak_600_800", "cluster": "parietal", "tmin": 0.600, "tmax": 0.800, "mode": "max"},
]


# -----------------------------------------------------------------------------
# Fonction Utilitaire (Helper Function)
# -----------------------------------------------------------------------------
def _window_mask(times: np.ndarray, tmin: float, tmax: float) -> np.ndarray:
    """
    Fonction utilitaire (helper) très rapide.
    
    Prend un array de temps (ex: [-0.2, -0.18, ..., 0.8]) et retourne
    un masque booléen (ex: [False, ..., True, True, ..., False])
    pour tous les points de temps situés entre tmin et tmax.
    
    Permet de sélectionner (slicer) les données EEG très efficacement.
    """
    return (times >= tmin) & (times <= tmax)

In [ ]:
# -----------------------------------------------------------------------------
# Fonctions de mesure sur les ERP
# -----------------------------------------------------------------------------
def peak_latency(evoked: mne.Evoked, 
                picks: list | str, 
                tmin: float, 
                tmax: float, 
                mode: str = 'max') -> tuple[float, float]:
    """
    Calcule le pic (amplitude et latence) sur un CLUSTER de canaux.

    Cette fonction moyenne d'abord les canaux du cluster en un seul
    "canal virtuel", PUIS trouve le pic (min ou max) sur ce signal moyen.
    
    Args:
        evoked: Les données EEG (objet MNE Evoked ou Epochs).
        picks: Liste de noms de canaux (ex: ['Pz', 'P3', 'P4']).
        tmin: Début de la fenêtre temporelle (en secondes).
        tmax: Fin de la fenêtre temporelle (en secondes).
        mode: 'max' (pic positif, ex: P3) ou 'min' (pic négatif, ex: N2).

    Returns:
        Un tuple (latence_en_secondes, amplitude_en_µV).
        Ex: (0.345, 4.51)
    """
    # 1. Copie et sélection des canaux et de la fenêtre de temps
    evk = evoked.copy().pick(picks)
    start, stop = evk.time_as_index([tmin, tmax])
            
    # 2. Slicer les données (data shape est [n_canaux, n_temps])
    segment = evk.data[:, start:stop]
    
    # 3. Calculer le "canal virtuel" en moyennant sur l'axe des canaux (axis=0)
    # On obtient un array 1D (le signal moyen du cluster)
    cluster_signal = segment.mean(axis=0)
    
    # 4. Trouver le pic (min ou max) sur ce signal 1D
    if mode == 'max':
        peak_idx = np.argmax(cluster_signal)
        peak_amp_V = np.max(cluster_signal)
    elif mode == 'min':
        peak_idx = np.argmin(cluster_signal)
        peak_amp_V = np.min(cluster_signal)
    else:
        raise ValueError("Mode doit être 'min' ou 'max'")

    # 5. Convertir l'index du pic en temps (secondes)
    # (L'index est relatif à 'start', donc on l'ajoute)
    peak_time_sec = evk.times[start + peak_idx]
    
    return float(peak_time_sec), float(peak_amp_V * 1e6)

def mean_amplitude_microvolt(evoked: mne.Evoked, 
                             picks: list | str, 
                             tmin: float, 
                             tmax: float) -> dict:
    """
    Calcule l'amplitude moyenne pour chaque canal d'un cluster ainsi que
    que au sein du cluster.

    Args:
        evoked: les données EEG (objet MNE Evoked ou Epochs).
        picks: Liste de noms de canaux (ex: ['Cz', 'Pz']) ou un seul (ex: 'Cz').
        tmin: Début de la fenêtre temporelle (en secondes).
        tmax: Fin de la fenêtre temporelle (en secondes).

    Returns:
        Un dictionnaire {canal: amplitude_moyenne_en_µV}.
        Ex: {'Fz': -1.23, 'Cz': -0.98}
    """
    # 1. Copie et sélection des canaux et de la fenêtre de temps
    evk = evoked.copy().pick(picks)
    start, stop = evk.time_as_index([tmin, tmax])

    # 3. Slicer les données (data shape est [n_canaux, n_temps])
    segment = evk.data[:, start:stop]
    
    # 4. Calculer la moyenne sur l'axe du temps (axis=1) pour chaque canal
    # et retourner un dictionnaire converti en microvolts (µV)
    return {
        ch: float(segment[i].mean() * 1e6)  # moyenne µV par canal
        for i, ch in enumerate(evk.ch_names)
    }

### Bloc Type 1 — extraction de métriques temporelles pour cw_incong sur les données 'evoked' 


In [ ]:
print("--- ANALYSE DES AMPLITUDES MOYENNES (Exploratoire) ---")
for config in ERP_MEAN_WINDOWS:
    cluster_name = config['cluster']
    picks = CHANNEL_CLUSTERS[cluster_name]
    
    # Appel de la fonction 1
    mean_amps = mean_amplitude_microvolt(
        evokeds['cw_incong'], 
        picks=picks, 
        tmin=config['tmin'], 
        tmax=config['tmax']
    )
    
    print(f"Caractéristique: {config['name']}")
    # Calcule la moyenne du dictionnaire pour avoir la moyenne du cluster
    avg_cluster_amp = np.mean(list(mean_amps.values()))
    print(f"  Moyenne du cluster '{cluster_name}': {avg_cluster_amp:.2f} µV")
    print(f"  Détail canaux : {mean_amps}")


print("\n--- ANALYSE DES PICS (Exploratoire) ---")
for config in ERP_PEAK_WINDOWS:
    cluster_name = config['cluster']
    picks = CHANNEL_CLUSTERS[cluster_name]
    
    # Appel de la fonction 2
    latency, amplitude = peak_latency(
        evokeds['cw_incong'],
        picks=picks,
        tmin=config['tmin'],
        tmax=config['tmax'],
        mode=config['mode']
    )
    
    print(f"Caractéristique: {config['name']}")
    print(f"  Pic du cluster '{cluster_name}': {amplitude:.2f} µV @ {latency * 1000:.0f} ms")

### Bloc Type 2 — Appliquer l'extraction de métriques temporelles sur les essais cw_incong

In [ ]:
# utilisez le meme code

### Bloc Type 3 — Explorer d'autres options

Pistes d'exploration :

* **Jouer avec les configurations :** Modifier les dictionnaires `CHANNEL_CLUSTERS`, `ERP_MEAN_WINDOWS`, et `ERP_PEAK_WINDOWS`.
* **Analyse par capteur :** Essayer d'analyser chaque capteur individuellement (sans utiliser de clusters).
* **Ajouter des composantes :** Identifier d'autres ERPs (ex: N1, P2) identifiés dans la littérature et les ajouter aux configurations.

#### Note: Inclure d'autre attributs, peut être une piste d'analyse additionelle comme cela permet de comparer les attribute de bases ainsi que d'autres.

## 4. Calculer les amplitudes et latences des epochs pour les utiliser comme attributs (features)

Maintenant que nous avons calculé et exploré visuellement les données **moyennées** (avec les objets `Evoked`), nous passons à l'extraction de ces attributs pour **tous les essais individuels** (avec l'objet `Epochs`).

L'objectif ici n'est plus de regarder la moyenne, mais de **quantifier** l'activité cérébrale pour **chaque essai individuellement**.

Nous allons réutiliser les mêmes dictionnaires de configuration (`CHANNEL_CLUSTERS`, `ERP_MEAN_WINDOWS`, `ERP_PEAK_WINDOWS`) et les fonctions d'aide (`mean_amplitude_epochs`, `peak_latency_epochs`) que nous avons définies précédemment.

**Le Résultat Final :** Le produit de cette étape sera un **DataFrame `pandas`** :
* Chaque **ligne** représentera un **essai (epoch)**.
* Chaque **colonne** représentera un **attribut** (ex: `P400_peak_amplitude_uV` ou `N400_mean_300_500`).

Ce DataFrame contiendra les données prêtes pour le Machine Learning : il servira à construire notre matrice `X` (les attributs) et notre vecteur cible `y` (les conditions, ou labels).

In [ ]:
def mean_amplitude_epochs(epochs: mne.Epochs, 
                          picks: list | str, 
                          tmin: float, 
                          tmax: float) -> np.ndarray:
    """
    Extrait l'amplitude moyenne d'un cluster sur une fenêtre de temps 
    pour CHAQUE ESSAI.

    Args:
        epochs: L'objet Epochs (données 3D : n_essais, n_canaux, n_temps).
        picks: Liste des canaux du cluster (ex: ['Pz', 'P3', 'P4']).
        tmin: Début de la fenêtre (secondes).
        tmax: Fin de la fenêtre (secondes).

    Returns:
        Un array 1D (shape [n_essais,]) contenant la moyenne en µV 
        de chaque essai.
    """
    # 1. Trouver les indices de temps
    start, stop = epochs.time_as_index([tmin, tmax])
    
    # 2. Obtenir les données 3D (n_essais, n_canaux_cluster, n_temps_fenetre)
    data = epochs.get_data(picks=picks)[:, :, start:stop]
    
    # 3. Calculer la moyenne sur les canaux (axis=1) ET le temps (axis=2)
    #    On obtient un array 1D (shape [n_essais,])
    mean_per_trial_V = data.mean(axis=(1, 2))
    
    # 4. Convertir en microvolts et retourner
    return mean_per_trial_V * 1e6

def peak_latency_epochs(epochs: mne.Epochs, 
                        picks: list | str, 
                        tmin: float, 
                        tmax: float, 
                        mode: str = 'max') -> tuple[np.ndarray, np.ndarray]:
    """
    Extrait le pic (amplitude et latence) d'un cluster sur une fenêtre 
    de temps pour CHAQUE ESSAI.

    Args:
        epochs: L'objet Epochs (données 3D).
        picks: Liste des canaux du cluster (ex: ['Pz', 'P3', 'P4']).
        tmin: Début de la fenêtre (secondes).
        tmax: Fin de la fenêtre (secondes).
        mode: 'max' (pic positif) ou 'min' (pic négatif).

    Returns:
        Un tuple de deux arrays 1D (chacun de shape [n_essais,]):
        (peak_latencies_sec, peak_amplitudes_uV)
    """
    # 1. Trouver les indices de temps et l'array de temps de la fenêtre
    start, stop = epochs.time_as_index([tmin, tmax])
    times_window = epochs.times[start:stop]
    
    # 2. Obtenir les données 3D (n_essais, n_canaux, n_temps_fenetre)
    data = epochs.get_data(picks=picks)[:, :, start:stop]
    
    # 3. Créer le "canal virtuel" pour chaque essai en moyennant les canaux
    #    Shape -> (n_essais, n_temps_fenetre)
    cluster_signals = data.mean(axis=1)
    
    # 4. Trouver les indices des pics (min ou max) sur l'axe du temps (axis=1)
    if mode == 'max':
        peak_indices = np.argmax(cluster_signals, axis=1)
    elif mode == 'min':
        peak_indices = np.argmin(cluster_signals, axis=1)
    else:
        raise ValueError("Mode doit être 'min' ou 'max'")

    # 5. Extraire les latences (en secondes) en utilisant les indices
    peak_latencies_sec = times_window[peak_indices]
    
    # 6. Extraire les amplitudes (en Volts) en utilisant les indices
    peak_amplitudes_V = np.array([
        cluster_signals[i, idx] for i, idx in enumerate(peak_indices)
    ])
    
    # 7. Convertir en µV et retourner
    return peak_latencies_sec, peak_amplitudes_V * 1e6

In [ ]:
import pandas as pd
import numpy as np

# 1. Initialiser le dictionnaire qui contiendra nos données
features_dict = {}

# Créer une map inversée {code: 'label'} pour traduire les événements
code_to_label = {v: k for k, v in epochs.event_id.items()}

# Ajouter la condition (ex: 'cv_cong') pour chaque essai
features_dict['condition'] = [code_to_label[code] for code in epochs.events[:, 2]]


# -----------------------------------------------------------------------------
# EXTRACTION DES CARACTÉRISTIQUES DE MOYENNE
# -----------------------------------------------------------------------------
print("  Extracting mean amplitude features...")

for config in ERP_MEAN_WINDOWS:
    name = config['name']
    cluster_name = config['cluster']
    picks = CHANNEL_CLUSTERS[cluster_name]
    tmin = config['tmin']
    tmax = config['tmax']
    
    # Appelle la fonction d'extraction sur les Epochs
    # 'mean_amps' sera un array 1D (shape [n_essais,])
    mean_amps = mean_amplitude_epochs(epochs, picks=picks, tmin=tmin, tmax=tmax)
    
    # Ajoute cet array comme une nouvelle colonne dans notre dictionnaire
    features_dict[name] = mean_amps
    print(f"    -> Feature '{name}' (cluster: {cluster_name}) ajoutée.")


# -----------------------------------------------------------------------------
# EXTRACTION DES CARACTÉRISTIQUES DE PIC
# -----------------------------------------------------------------------------
print("  Extracting peak features...")

for config in ERP_PEAK_WINDOWS:
    name = config['name']
    cluster_name = config['cluster']
    picks = CHANNEL_CLUSTERS[cluster_name]
    tmin = config['tmin']
    tmax = config['tmax']
    mode = config['mode']
    
    # Appelle la fonction d'extraction de pic sur les Epochs
    # 'latencies' et 'amplitudes' sont des arrays 1D
    latencies, amplitudes = peak_latency_epochs(
        epochs, picks=picks, tmin=tmin, tmax=tmax, mode=mode
    )
    
    # Ajoute *deux* nouvelles colonnes pour chaque configuration de pic
    features_dict[name + '_latency_sec'] = latencies
    features_dict[name + '_amplitude_uV'] = amplitudes
    print(f"    -> Features '{name}_latency' et '{name}_amplitude' (cluster: {cluster_name}) ajoutées.")


# -----------------------------------------------------------------------------
# CRÉATION DU DATAFRAME
# -----------------------------------------------------------------------------

# Convertit le dictionnaire de listes/arrays en un DataFrame pandas
df_features = pd.DataFrame(features_dict)

print("\n Extraction des caractéristiques terminée !")
print("\n--- Aperçu du DataFrame (df_features) ---")
print(df_features.head())

print("\n--- Informations sur le DataFrame ---")
df_features.info()


# -----------------------------------------------------------------------------
# Sauvegarde du DataFrame au format CSV
# -----------------------------------------------------------------------------
output_csv = deriv_analysis / f'sub-{subject}_session-{session}_run-{run}_erp_features.csv'
df_features.to_csv(output_csv, index=False)
print(f"\nDataFrame sauvegardé sous : {output_csv}")

### Bloc Type 3 — Explorer d'autres options (similaire au bloc Type 3 du 3.)

Pistes d'exploration :

* **Jouer avec les configurations :** Modifier les dictionnaires `CHANNEL_CLUSTERS`, `ERP_MEAN_WINDOWS`, et `ERP_PEAK_WINDOWS`.
* **Analyse par capteur :** Essayer d'analyser chaque capteur individuellement (sans utiliser de clusters).
* **Ajouter des composantes :** Identifier d'autres ERPs (ex: N200, P300) identifiés dans la littérature et les ajouter aux configurations.

#### Note: Inclure d'autre attributs, peut être une piste d'analyse additionelle comme cela permet de comparer les attribute de bases ainsi que d'autres.

## 5. Factoriser le pipeline pour tous les sujets

Objectifs : créer des fonctions modulaires qui calcule les attributs automatiquements pour tous les sujets. 

In [ ]:
def process_subject_features(subject: str, session: str, run: str, configs: dict, t_epoch: dict) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Exécute le pipeline complet (parsing, epoching, extraction) 
    pour un seul sujet.
    
    Retourne 3 DataFrames: 
        1. df_epoch_features (1 ligne par essai)
        2. df_evoked_features (1 ligne par condition)
        3. df_behavior (1 ligne par essai)

    Étapes :
        1. Chargement des données prétraitées.
        2. Parsing des événements.
        3. Création des époques (Epochs).
        4. Extraction des caractéristiques (features).
    
    Args:
        subject (str): ID du sujet (ex: '01').
        session (str): ID de la session (ex: '001').
        run (str): ID du run (ex: '01').
        configs (dict): Dictionnaire contenant les configurations 
                        (CHANNEL_CLUSTERS, ERP_MEAN_WINDOWS, ERP_PEAK_WINDOWS).
        t_epoch (dict): Dictionnaire des temps d'epoching (tmin, tmax, baseline).

    Returns:
        tuple: (df_evoked_features, df_epoch_features, df_behavior)
    """
    
    print(f"\n--- Traitement Sujet : {subject} ---")
    
    # --- 0. Décompression des configurations ---
    CHANNEL_CLUSTERS = configs['CHANNEL_CLUSTERS']
    ERP_MEAN_WINDOWS = configs['ERP_MEAN_WINDOWS']
    ERP_PEAK_WINDOWS = configs['ERP_PEAK_WINDOWS']

    # --- 1. Chargement des données ---
    raw = load_processed_raw(subject, session=session, run=run)

    # --- 2. Parsing des événements (Logique "Look-Ahead") ---
    print("  ... 1/6 Parsing des événements...")
    events, event_id = mne.events_from_annotations(raw)
    selected_event_id = {}
    for original, label in annotation_map.items():
        if np.str_(label) in event_id:
            selected_event_id[label] = event_id[np.str_(label)]

    print('Étiquettes retenues:')
    for label, code in selected_event_id.items():
        n_trials = int((events[:, 2] == code).sum())
        print(f' - {label:12s} → code {code:3d}, essais = {n_trials}')

    focus_event_id = {label: code for label, code in selected_event_id.items() if label in FOCUS_CONDITIONS}
    # --- 3. Création des Époques ---
    print("  ... 2/6 Création des époques...")
    epochs = mne.Epochs(raw, events, event_id=selected_event_id,
                        tmin=t_epoch['tmin'], tmax=t_epoch['tmax'],
                        baseline=t_epoch['baseline'], picks='eeg',
                        preload=True, detrend=None)

    # --- 4. Calcul des Evokeds (ERPs) ---
    print("  ... 3/6 Calcul des ERPs (Evokeds)...")
    evokeds = {}
    for label in focus_event_id:
        if len(epochs[label]) == 0:
            print(f'Avertissement: aucune epoch pour {label}.')
            continue
        evk = epochs[label].average()
        evokeds[label] = evk

    # --- 5. Extraction des Features (Evoked) ---
    print("  ... 4/6 Extraction des features (Evoked)...")
    evoked_features_list = []
    for condition, evk in evokeds.items():
        row_features = {'subject': subject, 'condition': condition}
        
        # Moyennes (Evoked)
        for config in ERP_MEAN_WINDOWS:
            picks = CHANNEL_CLUSTERS[config['cluster']]
            mean_amp = mean_amplitude_microvolt(evk, picks=picks, tmin=config['tmin'], tmax=config['tmax'])
            # on veut la moyenne du cluster
            avg_cluster_amp = np.mean(list(mean_amp.values()))
            row_features[config['name']] = avg_cluster_amp
            
        # Pics (Evoked)
        for config in ERP_PEAK_WINDOWS:
            picks = CHANNEL_CLUSTERS[config['cluster']]
            lat, amp = peak_latency(evk, picks=picks, tmin=config['tmin'], tmax=config['tmax'], mode=config['mode'])
            row_features[config['name'] + '_latency_sec'] = lat
            row_features[config['name'] + '_amplitude_uV'] = amp
            
        evoked_features_list.append(row_features)

    # --- 6. Extraction des Features (Epochs) ---
    print("  ... 5/6 Extraction des features (Epochs)...")
    epoch_features_dict = {}
    epoch_features_dict['subject'] = [subject] * len(epochs)
    code_to_label = {v: k for k, v in epochs.event_id.items()}
    epoch_features_dict['condition'] = [code_to_label[code] for code in epochs.events[:, 2]]

    # Moyennes (Epochs)
    for config in ERP_MEAN_WINDOWS:
        picks = CHANNEL_CLUSTERS[config['cluster']]
        mean_amps = mean_amplitude_epochs(epochs, picks=picks, tmin=config['tmin'], tmax=config['tmax'])
        epoch_features_dict[config['name']] = mean_amps

    # Pics (Epochs)
    for config in ERP_PEAK_WINDOWS:
        picks = CHANNEL_CLUSTERS[config['cluster']]
        latencies, amplitudes = peak_latency_epochs(
            epochs, picks=picks, tmin=config['tmin'], tmax=config['tmax'], mode=config['mode']
        )
        epoch_features_dict[config['name'] + '_latency_sec'] = latencies
        epoch_features_dict[config['name'] + '_amplitude_uV'] = amplitudes

    # --- 7. Création des DataFrames ---
    print("  ... 6/6 Création des DataFrames...")
    df_epoch_features = pd.DataFrame(epoch_features_dict)
    df_evoked_features = pd.DataFrame(evoked_features_list)
    
    print(f"  --- Sujet {subject} terminé. {len(df_epoch_features)} essais extraits. ---")
    
    return df_epoch_features, df_evoked_features

# --- 1. Définir les Constantes de l'Analyse ---
subjects_list = ['01', '02', '03', '04', '05', '06', '07', '08', '10', '11','13', '14']
session = '001'
run = '01'

# --- 2. Définir les Configurations de Features ---
CHANNEL_CLUSTERS = {
    "midline_fc": ["Fz", "Cz"],
    "centro_parietal": ["Cz", "P3", "P4"],
    "parietal": ["P3", "P4"],
    "occipital": ["O1", "O2"],
}
ERP_MEAN_WINDOWS = [
    {"name": "N400_mean_300_500_uV", "cluster": "centro_parietal", "tmin": 0.300, "tmax": 0.500},
    {"name": "P600_mean_600_800_uV", "cluster": "parietal", "tmin": 0.600, "tmax": 0.800},
]
ERP_PEAK_WINDOWS = [
    {"name": "N400_peak_300_500", "cluster": "centro_parietal", "tmin": 0.300, "tmax": 0.500, "mode": "min"},
    {"name": "P600_peak_600_800", "cluster": "parietal", "tmin": 0.600, "tmax": 0.800, "mode": "max"},
]
configs = {"CHANNEL_CLUSTERS": CHANNEL_CLUSTERS, "ERP_MEAN_WINDOWS": ERP_MEAN_WINDOWS, "ERP_PEAK_WINDOWS": ERP_PEAK_WINDOWS}
t_epoch = {"tmin": -0.2, "tmax": 0.8, "baseline": (-0.2, 0.0)}

# --- 3. Lancer la Boucle ---
all_epoch_features_dfs = []
all_evoked_features_dfs = [] # NOUVELLE liste

print(f"Lancement du pipeline pour {len(subjects_list)} sujets...")

for subject in subjects_list:
    df_epoch, df_evoked = process_subject_features(
        subject=subject, 
        session=session, 
        run=run,
        configs=configs,
        t_epoch=t_epoch
    )
    
    if not df_epoch.empty:
        all_epoch_features_dfs.append(df_epoch)
    if not df_evoked.empty:
        all_evoked_features_dfs.append(df_evoked) # NOUVEL append
            
# --- 4. Combiner et Sauvegarder les Résultats ---
print("\n--- Pipeline terminé. Combinaison des résultats... ---")

# Combiner tous les DataFrames
final_epoch_features_df = pd.concat(all_epoch_features_dfs, ignore_index=True)
final_evoked_features_df = pd.concat(all_evoked_features_dfs, ignore_index=True) # NOUVELLE concat

# Sauvegarder les fichiers CSV finaux
# (Assurez-vous que 'deriv_analysis' est défini)
output_epoch_csv = deriv_analysis / "all_subjects_epoch_erp_features.csv"
output_evoked_csv = deriv_analysis / "all_subjects_evoked_erp_features.csv" # NOUVEAU fichier

final_epoch_features_df.to_csv(output_epoch_csv, index=False)
final_evoked_features_df.to_csv(output_evoked_csv, index=False) # NOUVELLE sauvegarde

print(f"DataFrame de features (Epochs) final ({final_epoch_features_df.shape}):")
print(final_epoch_features_df.head())

print(f"\nDataFrame de features (Evoked) final ({final_evoked_features_df.shape}):") # NOUVEAU print
print(final_evoked_features_df.head())

print(f"\nRésultats sauvegardés dans : {deriv_analysis}")